In [29]:
import java.io.File

fun execute(
    builderBody: ProcessBuilder.() -> Unit
): Process = ProcessBuilder().apply {
        builderBody()
    }.start()


fun execute(
    command: List<String>,
    workingDirectory: File? = null
):List<String> = execute() {
    command(command)
    workingDirectory?.run {
        directory(this)
    }
}.run{
    val lines = inputStream.reader().readLines()
    println(lines)
    error(errorStream.reader().readText())

    return lines
}

fun execute(
    command: String,
    workingDirectory: File? = null,
):List<String> = execute(command.split(" "), workingDirectory)

## Start Bridge Docker

In [2]:
execute("git clone https://github.com/simplito/privmx-bridge-docker.git")

[]

## Start docker

In [43]:
import java.io.File

//API Key ID:  8ae66cd603fb1f28034f720e71d4ebc1
//        API Key Secret:  a05e094d5bf38a49bbe35fa862dcb34a
//PrivMX Bridge PubKey:  7ivTC2Cy694ZJ4AoXbP79bTrPv59TPJMunSfnd8anGUr7fiErP
//
//        IDs generated for your application:
//Solution ID:  1f13347a-ec33-430a-ae11-d03737065341
//Context ID:  a476d68a-4cda-4ab1-869e-d1bab7241ab6

 var bridgeUrl: String = "PrivMX Bridge URL:"
 var apiKey: String = "API Key ID:"
 var solutionId: String = "Solution ID:"


val setupOutput = execute("./setup.sh", File("tmp/privmx-bridge-docker"))

//bridgeUrl = setupOutput.filter { it.trim().startsWith("PrivMX Bridge URL:") }.get(0).substringAfter("PrivMX Bridge URL:").replace(" ", "")
//apiKey = setupOutput.filter { it.trim().startsWith("API Key ID:") }.get(0).substringAfter("API Key ID:").replace(" ", "")
//solutionId = setupOutput.filter { it.trim().startsWith("Solution ID:") }.get(0).substringAfter("Solution ID:").replace(" ", "")

//

bridgeUrl = setupOutput.filter { it.trim().startsWith(bridgeUrl) }.get(0).substringAfter(bridgeUrl).replace(" ", "")
apiKey = setupOutput.filter { it.trim().startsWith(apiKey) }.get(0).substringAfter(apiKey).replace(" ", "")
solutionId = setupOutput.filter { it.trim().startsWith(solutionId) }.get(0).substringAfter(solutionId).replace(" ", "")



[***************************************************************************, *                                                                         *, *                         PRIVMX BRIDGE INSTALLER                         *, *                                                                         *, ***************************************************************************, , -----------------, Booting up, -----------------, OK, , -----------------, Preparing data, -----------------, API key already created OK, Solution already created OK, Context already created OK, , ***************************************************************************, *                                                                         *, *     _____      _       __  ____   __  ____       _     _                *, *    |  __ \    (_)     |  \/  \ \ / / |  _ \     (_)   | |               *, *    | |__) | __ ___   _| \  / |\ V /  | |_) |_ __ _  __| | __ _  ___     *, *    |  ___/ '__| \ \ / / |\/|

java.lang.IllegalStateException:  privmx-bridge Pulling 
 privmx-bridge Pulled 
 Container privmx-bridge-docker-mongodb-1  Running
 Container privmx-bridge-docker-privmx-bridge-1  Running
 Container privmx-bridge-docker-privmx-bridge-1  Waiting
 Container privmx-bridge-docker-mongodb-1  Waiting
 Container privmx-bridge-docker-privmx-bridge-1  Healthy
 Container privmx-bridge-docker-mongodb-1  Healthy


In [4]:
var pubKey: String = ""
var privKey: String = ""

fun generateKeyPair(){
    val genKeyOutput = execute("./genKeyPair.sh", File("privmx-bridge-docker"))
    pubKey = genKeyOutput.filter { it.trim().startsWith("PUB_58=") }.get(0).substringAfter("PUB_58=").replace(" ", "")
    privKey = genKeyOutput.filter { it.trim().startsWith("PRIV_WIF=") }.get(0).substringAfter("PRIV_WIF=").replace(" ", "")
}

In [5]:
import kotlinx.serialization.Serializable
import kotlinx.serialization.json.JsonArray
import kotlinx.serialization.json.jsonArray
import kotlinx.serialization.decodeFromString
import kotlinx.serialization.json.Json
import kotlinx.serialization.json.JsonElement

@Serializable
data class Data(val result: ResultData)

@Serializable
data class ResultData(val list: List<ItemData>)

@Serializable
data class ItemData(val id: String, val name: String, val policy: Map<String, String>)

In [6]:
data class BridgeUser(val id: String, val pubKey: String, val privKey: String)

In [7]:
fun addUser(context: String, userId: String, userPubKey: String,) {
    execute(
        listOf(
            "./cli.sh",
            "context/addUserToContext",
            "{\"contextId\": \"$context\", \"userId\": \"$userId\", \"userPubKey\": \"$userPubKey\"}"
        ),
        File("privmx-bridge-docker")
    )
}

In [8]:
import com.simplito.kotlin.privmx_endpoint_extra.model.SortOrder

fun listContexts(skip: Int = 0, limit: Int = 10, sortOrder: String = SortOrder.DESC): List<String> {
    val resultList: List<String> = execute(
        listOf(
            "./cli.sh",
            "context/listContexts",
            "{\"skip\": $skip, \"limit\": $limit, \"sortOrder\": \"$sortOrder\"}"
        ),
        File("privmx-bridge-docker")
    )

    val json = resultList.joinToString("\n")
    val contexts = Json {
        ignoreUnknownKeys = true
    }.decodeFromString<Data>(json).result.list.map {
        println(it)
        it.id
    }

    return contexts
}

In [9]:
lateinit var user1: BridgeUser
lateinit var user2: BridgeUser

val contextId = listContexts().get(0)
println("ContextId: $contextId")

generateKeyPair()
addUser(contextId, "user_1", pubKey)
user1 = BridgeUser("user_1", pubKey, privKey)

generateKeyPair()
addUser(contextId, "user_2", pubKey)
user2 = BridgeUser("user_2", pubKey, privKey)

ItemData(id=97961888-449b-474e-b0d0-ef3ace9e04dc, name=MainContext, policy={})
ContextId: 97961888-449b-474e-b0d0-ef3ace9e04dc


## Create instance of PrivMX Endpoint Container

In [10]:
import com.simplito.kotlin.privmx_endpoint_extra.lib.PrivmxEndpointContainer
import com.simplito.kotlin.privmx_endpoint_extra.model.Modules

val initModules = setOf(
    Modules.THREAD, // initializes ThreadApi to working with Threads
    Modules.STORE, // initializes StoreApi to working with Stores
    Modules.INBOX // initializes InboxApi to working with Inboxes
)
val endpointContainer = PrivmxEndpointContainer()

## Connect to platform

In [11]:
val endpointSession = PrivmxEndpointContainer().connect(
    initModules,
    user1.privKey,
    solutionId,
    bridgeUrl
)

//todo - czy powinno się móc dodać kvdbApi np? jesli nie dodalam tego modulu do initModules - endpointSession.kvdbApi

In [12]:
@Serializable
data class ThreadPublicMeta(val tags: List<String>)

In [13]:
import com.simplito.kotlin.privmx_endpoint.model.UserWithPubKey
import kotlinx.serialization.Serializer
import kotlinx.serialization.encodeToString

val users: List<UserWithPubKey> = listOf(
    UserWithPubKey(user1.id, user1.pubKey),
    UserWithPubKey(user2.id, user2.pubKey)
)
val managers: List<UserWithPubKey> = listOf(
    UserWithPubKey(user1.id, user1.pubKey)
)
val threadNameAsPrivateMeta = "Thread Name"
val publicMeta = ThreadPublicMeta(
    listOf("TAG1", "TAG2", "TAG3")
)

val threadId = endpointSession.threadApi?.createThread(
    contextId,
    users,
    managers,
    Json.encodeToString(publicMeta).encodeToByteArray(),
    threadNameAsPrivateMeta.encodeToByteArray()
)

In [ ]:
// Connect as user2
val endpointSessionUser2 = PrivmxEndpointContainer().connect(
    initModules,
    user2.privKey,
    solutionId,
    bridgeUrl
)

In [14]:
val publicMeta = ByteArray(0)
val privateMeta = ByteArray(0)

// user1 sends message
endpointSession.threadApi?.sendMessage(
    threadId,
    publicMeta,
    privateMeta,
    "Message 1".encodeToByteArray()
)

686d1fdf8e6e72dbd03aadb2

In [15]:
endpointSessionUser2.threadApi?.sendMessage(
    threadId,
    publicMeta,
    privateMeta,
    "Message 2".encodeToByteArray()
)

In [40]:
val pagingList = endpointSession.threadApi?.listMessages(
    threadId,
    0,
    100,
    SortOrder.ASC
)
val threadName: String = String(endpointSession.threadApi?.getThread(threadId)?.privateMeta!!)

println("---|$threadName|---")

pagingList?.readItems?.forEach {
    val author = it.info.author
    val message = String(it.data)

    println("[$author]: $message")
}

---|Thread Name|---
[user_1]: Message 1
[user_2]: Message 2


In [42]:
// version 1
endpointContainer.disconnect(endpointSession.connection.getConnectionId()!!)
endpointContainer.disconnect(endpointSessionUser2.connection.getConnectionId()!!)
endpointContainer.close()

// version 2
endpointContainer.disconnectAll()
endpointContainer.close()

// version 3
endpointContainer.close()

java.lang.NullPointerException: 